<p style="font-size: 40px; text-align: center;">Efficient Use of Cluster Misha - Part 2</p>


## 1. Manage Disk Space

Having enough free space in your home directory is crucial for running Open OnDemand and many applications, since they need to write cached files there. If your home is full — either because the quota limit or the file limit has been reached — applications will fail, and Open OnDemand interactive apps won't start. 

One typical culprit residing in the home directory that takes up a lot of space and uses a lot of file count is Conda. By default, your Conda packages are stored in `$HOME/.conda/pkgs` and your Conda environments are installed in `$HOME/.conda/envs`. When you run into issues with quota, a practical solution is to move everything in `$HOME/.conda` to a different non-temporary space (usually your project or a PI fileset), and then link it back to `$HOME/.conda`. Doing so, you resolve the quota issue without breaking your Conda environments. 

It usually takes hours to move the entire `$HOME/.conda`. We created a simple shell script `mv_conda.sh` to make the move more convinient. 

<div class="alert alert-block alert-warning">
    <b>Warning:</b> refine mv_conda.sh
</div>

In [ ]:
du -h -d 2 $HOME/.conda
nohup mv_conda.sh >& mv_conda.log &

Open OnDemand may also take up a lot of space in your home directory, especially if you use Job Composor or RStudio. 

In [ ]:
ycrc_clean_rstudio.sh -f

## 2. Understand Advanced SLURM Options 

In addition to the commonly used SLURM options covered in part one of the tutorial, several less-used but useful options can affect memory allocation and task placement. To understand these options, let's first review the following terminology. 

* <b>CPU (Central Processing Unit)</b> — The main processor of a computer that executes instructions and performs calculations. Each CPU may contain multiple cores, enabling parallel computation.
* <b>Socket</b> — The physical interface on a node’s motherboard that holds a CPU. A node may have one or more sockets, each occupied by a multi-core CPU.
* <b>Core or Processor</b> — An individual processing unit within a CPU capable of executing its own thread of instructions. Multiple cores allow a CPU to run multiple operations in parallel.
* <b>GPU (Graphics Processing Unit)</b> — A specialized processor designed for highly parallel computations, often used to accelerate deep learning, scientific simulations, and other compute-intensive tasks.
* <b>Job</b> — A user-submitted workload managed by the SLURM scheduler. A job may consist of one or more tasks that execute on allocated nodes, CPUs, or GPUs according to the requested resources.
* <b>Task</b> — A single unit of work or process within a SLURM job. Each task usually runs one instance of a program and may execute on a separate CPU core or GPU.
* <b>Process</b> — An instance of a running program with its own memory space and system resources. In parallel computing, multiple processes can run concurrently and communicate through interfaces such as MPI.
* <b>Thread</b> — The smallest unit of execution within a process. Threads share the same memory and resources of their parent process, enabling lightweight parallelism.
* <b>Node</b> — A single physical or virtual machine within a cluster. Each node contains its own CPUs, memory, storage, and often GPUs, and is managed by SLURM to execute jobs and tasks.

| Category|   Options  |
|--------|----------------------------------|
|Memory  |  --mem=\<size>[K\|M\|G\|T] <br> --mem-per-cpu=\<size>[K\|M\|G\|T] <br> --mem-per-gpu=\<size>[K\|M\|G\|T] |
|Tasks   | --ntasks=\<ntasks> <br>--ntasks-per-node=\<ntasks> <br> --ntasks-per-core=\<ntasks> <br> --ntasks-per-gpu=\<ntasks> |
|GPUs     |  --gpus=[type:]\<ngpus> <br> --gres=gpu:[type:]\<ngpus> <br> --gpus-per-task=[type:]\<ngpus> <br> --gpus-per-node=[type:]\<ngpus> |
|Cores|  --cpus-per-task=\<ncpus> <br> --cpus-per-gpu=\<ncpus>   |
|Socket  | --cores-per-socket=\<number> <br> --gpus-per-socket=[type:]\<number> <br> --ntasks-per-socket=\<number> <br> --sockets-per-node=\<number> |

### Choose right options for OpenMP, MPI, and hybrid jobs
Parallel applications running on a cluster generally fall into three categories: OpenMP, MPI, and hybrid (OpenMP+MPI). Depending on the types of applications you run, the options will differ slightly. The following three diagrams illustrate how to use the related options for each type. 


<center><img src="figures/OpenMP.png" width="400"></center>
<center> Example of a four-thread OpenMP run </center>
<br>
<center><img src="figures/MPI-wrong.png" width="400"></center>
<center> Example of a four-process MPI run </center>
<br>
<center><img src="figures/hybrid.png" width="400"></center>
<center> Example of a 2(thread)x2(process) hybrid run </center>

### Memory size and core count

When using a combination of the options listed in the table above, pay attention to the total memory and total core count requested for your job. It's easy to make an unintentional request that far exceeds your intended amount and causes scheduling issues. Let's see two examples below.
<pre>
--mem-per-cpu=64g 
--cpus-per-gpu=8
--gpus = 1
</pre>
<pre> 
aa
</pre>


In [ ]:
# Run the following command in a bash shell
# salloc --cpus-per-gpu=8 --gpus=1 --mem=64g -w r4518u01n01 -p gpu

In [ ]:
# Run the following command in a bash shell
# salloc --cpus-per-gpu=8 --gpus=1 --mem=64g -w r4518u01n01 -p gpu --tasks=8


### Difference between --gpus and --gres=gpu

The `--gpus=[type:]<ngpus>` option requests total `ngpus` GPUs for the job. It is typically used with the following GPU-aware options for GPU distribution.
* `--gpus-per-node`
* `--gpus-per-task`
* `--gpus-per-socket`

The `--gres=gpu:[type:]<number>` option requests `ngpus` GPUs <b>per node</b>. 

`--gpus=[type:]<ngpus>` is available in newer versions of SLURM and is generally preferred over `--gres=gpu:[type:]<number>`. However, a common pitfall when using `--gpus` is requesting multiple nodes without specifying GPU-aware options (such as `--gpus-per-node` or `--gpus-per-task`) to control how GPUs are distributed. This often leads to an uneven or unbalanced GPU allocation across nodes.

In [ ]:
salloc -p gpu --gpus=4 -N 2 -n 2 --ntasks-per-node=1

## 3. QOS-Based GPU Scheduling Protocol

Misha uses a single GPU partition that includes both institute-purchased and PI-purchased nodes. Compared with maintaining separate PI partitions, a unified partition improves GPU scheduling efficiency and sharing. It also reduces resource fragmentation, as the scheduler can pack smaller jobs onto the same node whenever possible.

Although there is only one partition, each research group can still access the <b>GPU devices</b> purchased by their PI through their <b>group-specific QOS</b>. We emphasize GPU devices rather than nodes because the <b>type</b> and <b>number</b> of GPUs are what users care about most. For example, if a PI purchases two nodes, each equipped with four H100 cards, their main concern is being able to use any of the eight H100 GPUs as needed—regardless of which nodes they reside on or whether they are on the same node. This principle underpins Misha’s QOS-based GPU scheduling model within a single, shared GPU partition.

### How to use a group QOS

A SLURM QOS tag, for the purpose of defining a group's GPU resource, takes the format `qos_groupname`. This tag defines the types of GPU devices a group can access and the total number of each kind. 

To access a group QOS, `--qos=qos_groupname` must be accompanied with `--gres=gpu:type:num` or `--gpus=type:num`. 

You can run the following command to check which group QOS tags you have access to and what GPU resources these tags contain. 

In [ ]:
gpu_check qos --me

In [ ]:
gpu_check alloc

In [ ]:
gpu_check

### Use `gpu_check`to help choosing GPU

In [ ]:
cat test.job

In [ ]:
gpu_check

In [ ]:
sbatch --

<div class="alert alert-block alert-warning">
    <b>Warning:</b> Run gpu_check periodically to get an instant view about GPU allocation. As there is a cost associated with it, do not run it frequently. Also, it currently only accounts for running jobs, not jobs in the queue.  
</div>

## 4. Integrate Conda, Jupyter, VSCode, and GitHub

Setting up your local VSCode to work with a remote Jupyter Conda environment may be confusing for first-time users.  But once a remote development workflow is configured, you’ll appreciate how convenient and seamless it becomes. Follow the steps below.

1. Access ood-misha.ycrc.yale.edu and start an instance of VSCode Proxy: click 'Interactive Apps' -> 'VSCode Proxy'.
2. Once VSCode Proxy starts running, it shows how to configure the local SSH config file. Set up your local SSH config file as displayed on the session card.
        <center><img src=figures/vscode-proxy-session-card.png width=600> </center>


4. Launch VSCode from your local machine. Install the SSH extension. Once done, the 'Remote Explorer' icon will appear on the left sidebar. Click it, you will see one entry for 'jump-server' and one entry for 'vscode-server'. Click 'Connect in Current Window' or 'Connect in New Window' next to 'vscode-server'. VSCode starts connecting to the remote compute node where VSCode Proxy is launched. 
5. Click 'Command Center' at the top, and you will be prompted to connect to the remote node. Follow the prompts, including responding to the DUO authentication request. 
        <center><img src=figures/vscode-duo.jpg width=800> </center>
6. Once done, open a terminal and confirm that you are now connected to the Misha compute node where the VSCode Proxy instance is running. This indicates that a connection has been established. 
7. Install the Jupyter extension for Remote SSH. 
8. Create a new notebook or open an existing one. Click 


Click "Select Kernel" -> click "Select Python Environment" -> select 
The kernel selected is for Python. If you want to switch to a different kernel, 
Select Kernel ->  Select Another Kernel -> Jupyter Kernel -> Select a Jupyter Kernel (type in the path to python)





The above integration is simple if your local machine runs Linux or macOS. If your local machine runs Windows, please contact ping.luo@yale.edu for assistance. 

chatbot example

## 5. Checkpoint and Restart

Jobs that don't use a group QoS or `qos_nmi` have the greatest flexibility in using any available GPUs on Misha. However, the trade-off is that these jobs are subject to preemption. The SLURM scheduler on Misha provides a one-hour grace period for preempted jobs.  When a job is selected for preemption, SLURM sends it a SIGTERM. If the job catches this signal, it can use the grace period to complete checkpointing and shut down cleanly. Otherwise, if the job does not handle SIGTERM, it will be terminated immediately, which may result in loss of progress or data.

### Catch SIGTERM in user program
The Python example below demonstrates how to handle SIGTERM within a program. However, if you run the program directly in a batch script, it will not catch the SIGTERM signal sent by SLURM during preemption. To ensure the signal is properly received and handled, you should run the program using `srun` within the job script.
```<phthon>
#!/usr/bin/env python

import signal
import sys
import time

SIGTERM = 15
def signal_handler(sig, frame):
    print('A signal has been caught.')
    time.sleep(10)

signal.signal(SIGTERM, signal_handler)
time.sleep(600)
print('Done')
```

For non-priority jobs that do not use a group QoS or `qos_nmi`, the key takeaways are:
1. Handle SIGTERM in your program so it can shut down gracefully. 
2. Launch your program with `srun` in your job script to ensure it receives the SIGTERM signal correctly.


### Automatic checkpoint and restart

The SLURM job option `--requeue` indicates that a batch job is eligible to be requeued. If your program implements checkpointing by catching SIGTERM, as described above, and automatically resumes from the last checkpoint when restarted in the same directory, then adding `--requeue` to your job script enables fully automatic checkpoint–restart integration.

Let's see an example of checkpoint-restart. As requeuing happens when preemption The example is adapted from https://github.com/rasbt/LLMs-from-scratch. 

In [ ]:
cd code/llm-from-scratch
cat gpt_train.py
cat gpt_train.job

In [ ]:
# remove the old checkpoint.pt
rm -f checkpoint.pt
# Copy and run the following command from a shell session outside of Jupyter.
# jobid=$(sbatch gpt_train.job|cut -d' ' -f4); echo $jobid >jobid.out
jobid=$(cat jobid.out)
while [ 1 ]; do
    squeue --stat=R|grep $jobid
    if [ $? == 0 ]; then
        break
    fi
done
# after the job runs for 20 seconds, cancel the job. 
# The job won't be requeued because it was canceled by scancel. 
# If it is preempted, it will be requeued. 
sleep 20
scancel $jobid
# The checkpoint file is generated
ls -l checkpoint.pt
rm jobid.out

In [ ]:
# Resubmit gpt_train.job. Copy 
# jobid=$(sbatch gpt_train.job|cut -d' ' -f4); echo $jobid > jobid.out
while [ 1 ]; do
    squeue --stat=R|grep $jobid
    if [ $? == 0 ]; then
        break
    fi
done
# after the job starts running for 20 seconds, cancel the job. 
# The job won't be requeued because of its termination using scancel. 
# If it is preempted, it will be requeued. 
sleep 10
# When the job is running, check its output file, and you will see it resume training from the last checkpointed epoch.
cat slurm-${jobid}.out

## 6. Monitor Job Performance

### Node Status and Job Status

<center><img src=figures/node-status-inefficiency.png width=300></center>

Once you have a job running on the cluster, you can SSH into any of the nodes where your job is running.

Job inefficiency: low GPU utilization; idle GPU; low VRAM usage

For CPU jobs: jobstats jobid

### GPU job performance alerts

GPU jobs are monitored 24x7.
Email alerts sent to inefficient jobs every day at 5pm
    Jobs running less than two hours 

### Benchmarks and make informed GPU choices

llm example